# PTM Binder Workshop: Multi-design sweep around the CD3e ITAM motif

This notebook keeps the original PTM-target setup, but it samples multiple RFD3 backbones, runs LigandMPNN on every backbone, and then refolds every designed complex with RF3 so you can compare the whole candidate set side by side.


## 0. Setup

This notebook works in both Google Colab and a local Pixi setup.

- In Colab: switch to a GPU runtime and run the setup cells from the top.
- Locally: launch Jupyter from the `ptm_foundry` repo in the Pixi `dev` environment.

Recommended local setup from the repo root:

```bash
pixi install -e dev
pixi run -e dev install-workshop-kernel
pixi run -e dev workshop-notebook
```

Then open this notebook with the `PTM Workshop` kernel and run it from the top.


In [1]:
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
GIT_URL = os.environ.get("PTM_FOUNDRY_GIT_URL", "https://github.com/magnusbauer/ptm_foundry.git")
GIT_REF = os.environ.get("PTM_FOUNDRY_GIT_REF", "workshop")

if IN_COLAB:
    REPO_DIR = Path("/content/foundry")
else:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent]
    REPO_DIR = next(
        (candidate for candidate in candidates if (candidate / "examples").exists() and (candidate / "models").exists()),
        cwd,
    )

if IN_COLAB and not REPO_DIR.exists():
    subprocess.check_call(
        ["git", "clone", "--branch", GIT_REF, GIT_URL, str(REPO_DIR)]
    )

SOURCE_PATHS = [
    REPO_DIR / "src",
    REPO_DIR / "models" / "rfd3" / "src",
    REPO_DIR / "models" / "mpnn" / "src",
    REPO_DIR / "models" / "rf3" / "src",
    REPO_DIR / "examples",
]
for source_path in SOURCE_PATHS:
    if source_path.exists() and str(source_path) not in sys.path:
        sys.path.insert(0, str(source_path))

print(f"Running in Colab: {IN_COLAB}")
print(f"Repository root: {REPO_DIR}")
if IN_COLAB:
    print(f"Clone source:    {GIT_URL} @ {GIT_REF}")


Running in Colab: False
Repository root: /net/scratch/magnusb/43_workshop/ptm_foundry


In [2]:
SPOOF_CIF_PATH = REPO_DIR / "examples" / "spoof_cif.py"
WORKSHOP_OUTPUT_DIR = REPO_DIR / "examples" / "workshop_outputs"

if not SPOOF_CIF_PATH.exists():
    raise FileNotFoundError(
        f"Expected workshop helper at {SPOOF_CIF_PATH}. "
        "Make sure the repo checkout includes examples/spoof_cif.py."
    )

WORKSHOP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Using workshop helper: {SPOOF_CIF_PATH}")
print(f"Workshop outputs:     {WORKSHOP_OUTPUT_DIR}")


Using workshop helper: /net/scratch/magnusb/43_workshop/ptm_foundry/examples/spoof_cif.py
Workshop outputs:     /net/scratch/magnusb/43_workshop/ptm_foundry/examples/workshop_outputs


In [3]:
%%time

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

os.environ["CCD_MIRROR_PATH"] = ""
os.environ["PDB_MIRROR_PATH"] = ""

for env_name, default_value in {
    "DEBUG": "0",
    "TYPE_CHECK": "0",
    "NAN_CHECK": "1",
    "DISABLE_CUEQUIVARIANCE": "0",
}.items():
    if not os.environ.get(env_name, "").strip():
        os.environ[env_name] = default_value

CKPT_DIR = Path.home() / ".foundry" / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
os.environ["FOUNDRY_CHECKPOINTS_DIR"] = str(CKPT_DIR)


def ensure_pip() -> None:
    if importlib.util.find_spec("pip") is None:
        subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])


def pip_install(packages: list[str]) -> None:
    if not packages:
        return
    ensure_pip()
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


def start_download(url: str, dest: Path) -> tuple[subprocess.Popen, Path]:
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp_dest = dest.with_suffix(dest.suffix + ".part")
    if tmp_dest.exists():
        tmp_dest.unlink()
    proc = subprocess.Popen(
        [
            "curl",
            "-L",
            "--fail",
            "--retry",
            "3",
            "-o",
            str(tmp_dest),
            url,
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    return proc, tmp_dest


if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"],
        check=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

REQUIRED_PACKAGES = {
    "matplotlib": "matplotlib",
    "networkx": "networkx",
    "pandas": "pandas",
    "hydride": "hydride",
    "biotite": "biotite",
    "atomworks": "atomworks[ml]>=2.1.1",
    "lightning": "lightning>=2.5.0",
    "rootutils": "rootutils>=1.0.7,<1.1",
    "hydra": "hydra-core>=1.3.0,<1.4",
    "environs": "environs>=11.0.0,<12",
    "rich": "rich>=13.9.4",
    "jaxtyping": "jaxtyping>=0.2.17,<1",
    "beartype": "beartype>=0.18.0,<1",
    "loralib": "loralib>=0.1.1",
    "einops": "einops>=0.8.0,<1",
    "einx": "einx>=0.1.0,<1",
    "opt_einsum": "opt_einsum>=3.4.0,<4",
    "tree": "dm-tree>=0.1.6,<1",
    "zstandard": "zstandard",
    "toolz": "toolz",
    "pydantic": "pydantic>=2.8",
    "plotly": "plotly>=5,<7",
    "anywidget": "anywidget>=0.9,<1",
    "ipywidgets": "ipywidgets>=8,<9",
    "ipymolstar": "ipymolstar>=0.1,<0.2",
    "assertpy": "assertpy",
}
missing_packages = [
    package
    for module_name, package in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    print("Installing missing packages:")
    for package in missing_packages:
        print(" -", package)
    pip_install(missing_packages)
else:
    print("Python dependencies already satisfied.")

if IN_COLAB:
    try:
        from google.colab import output as colab_output

        colab_output.enable_custom_widget_manager()
        print("Enabled Colab custom widget manager for Mol*.")
    except Exception as error:
        print(f"Warning: could not enable Colab custom widgets: {error}")

CHECKPOINTS = {
    "rfd3": {
        "url": "https://files.ipd.uw.edu/pub/rfd3/rfd3_foundry_2025_12_01_remapped.ckpt",
        "filename": "rfd3_latest.ckpt",
    },
    "ligandmpnn": {
        "url": "https://files.ipd.uw.edu/pub/ligandmpnn/ligandmpnn_v_32_010_25.pt",
        "filename": "ligandmpnn_v_32_010_25.pt",
    },
    "rf3": {
        "url": "https://files.ipd.uw.edu/pub/rf3/rf3_foundry_01_24_latest_remapped.ckpt",
        "filename": "rf3_foundry_01_24_latest_remapped.ckpt",
    },
}

download_jobs = []
for name, info in CHECKPOINTS.items():
    dest = CKPT_DIR / info["filename"]
    if dest.exists():
        print(f"{name}: already present at {dest}")
        continue
    print(f"Starting download {name} -> {dest}")
    proc, tmp_dest = start_download(info["url"], dest)
    download_jobs.append((name, dest, tmp_dest, proc))

for name, dest, tmp_dest, proc in download_jobs:
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"Download failed for {name} with exit code {rc}")
    tmp_dest.replace(dest)
    print(f"{name}: downloaded to {dest}")

print("\nCheckpoint directory contents:")
for checkpoint_path in sorted(CKPT_DIR.iterdir()):
    print(" -", checkpoint_path.name)

RFD3_CKPT = CKPT_DIR / CHECKPOINTS["rfd3"]["filename"]
LIGANDMPNN_CKPT = CKPT_DIR / CHECKPOINTS["ligandmpnn"]["filename"]
RF3_CKPT = CKPT_DIR / CHECKPOINTS["rf3"]["filename"]

print("\nResolved checkpoint paths:")
print(f" - RFD3:       {RFD3_CKPT}")
print(f" - LigandMPNN: {LIGANDMPNN_CKPT}")
print(f" - RF3:        {RF3_CKPT}")


Python dependencies already satisfied.
rfd3: already present at /home/magnusb/.foundry/checkpoints/rfd3_latest.ckpt
ligandmpnn: already present at /home/magnusb/.foundry/checkpoints/ligandmpnn_v_32_010_25.pt
rf3: already present at /home/magnusb/.foundry/checkpoints/rf3_foundry_01_24_latest_remapped.ckpt

Checkpoint directory contents:
 - ligandmpnn_v_32_010_25.pt
 - proteinmpnn_v_48_020.pt
 - rf3_foundry_01_24_latest_remapped.ckpt
 - rfd3_latest.ckpt

Resolved checkpoint paths:
 - RFD3:       /home/magnusb/.foundry/checkpoints/rfd3_latest.ckpt
 - LigandMPNN: /home/magnusb/.foundry/checkpoints/ligandmpnn_v_32_010_25.pt
 - RF3:        /home/magnusb/.foundry/checkpoints/rf3_foundry_01_24_latest_remapped.ckpt
CPU times: user 2.19 ms, sys: 1.23 ms, total: 3.43 ms
Wall time: 17.4 ms


In [4]:
import importlib
import json
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", module="atomworks")

EXAMPLES_DIR = REPO_DIR / "examples"
if str(EXAMPLES_DIR) not in sys.path:
    sys.path.insert(0, str(EXAMPLES_DIR))

from atomworks.constants import PROTEIN_BACKBONE_ATOM_NAMES
from atomworks.io.utils.io_utils import to_cif_file
from atomworks.io.utils.visualize import view
from biotite.structure import rmsd, superimpose
from lightning.fabric import seed_everything
from spoof_cif import (
    chain_summary,
    extract_chain_sequence,
    plot_residue_bond_graph,
)

from atomworks.constants import DICT_THREE_TO_ONE, UNKNOWN_AA
from atomworks.io.tools.inference import (
    build_msa_paths_by_chain_id_from_component_list,
    components_to_atom_array,
)
from atomworks.io.utils.io_utils import to_cif_file

import plotly.express as px

import ptm_workshop as ptm_workshop_module

ptm_workshop_module = importlib.reload(ptm_workshop_module)

from ptm_workshop import (
    PHOSPHATE_ATOMS,
    align_mobile_on_binder_backbone,
    build_rf3_metrics_df,
    build_rfd3_metrics_df,
    compute_phosphosite_hbond_metrics,
    compute_selection_sasa_metrics,
    extract_min_interface_pae,
    make_browser_structure_spec,
    make_structure_browser,
    make_studio_metric_browser,
    make_structure_overlay_spec,
    paired_common_indices,
    rmsd_for_masks,
)

print(f"Reloaded ptm_workshop from {ptm_workshop_module.__file__}")


Reloaded ptm_workshop from /net/scratch/magnusb/43_workshop/ptm_foundry/examples/ptm_workshop.py


## 1. Define the Workshop Target

The peptide target is from an ITAM motif from CD3ε `PVPNPD(PTR)EPIRKGQ`, where `(PTR)` is phosphotyrosine. We are going to build a binder around that site, so first let's make sure the residue numbering and PTM position line up the way we expect.


In [5]:
TARGET_SEQUENCE = "PVPNPD(PTR)EPIRKGQ"
TARGET_CHAIN_ID = "B"
BINDER_LENGTH = 100
EXAMPLE_NAME = "ptr_workshop_mult"
RFD3_DIFFUSION_BATCH_SIZE = 3
RFD3_N_BATCHES = 1
MPNN_SEQUENCES_PER_BACKBONE = 4
RF3_DIFFUSION_BATCH_SIZE = 1
WORK_DIR = Path("/content/ptm_workshop_mult") if IN_COLAB else WORKSHOP_OUTPUT_DIR / "mult"
WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f"Target sequence: {TARGET_SEQUENCE}")
print(f"Target chain:    {TARGET_CHAIN_ID}")
print(f"Binder length:   {BINDER_LENGTH}")
print(f"Working dir:     {WORK_DIR}")
print(f"RFD3 sweep:      diffusion_batch_size={RFD3_DIFFUSION_BATCH_SIZE}, n_batches={RFD3_N_BATCHES}")
print(f"LigandMPNN:      {MPNN_SEQUENCES_PER_BACKBONE} sequences per backbone")
print(f"RF3 sweep:       diffusion_batch_size={RF3_DIFFUSION_BATCH_SIZE}")


Target sequence: PVPNPD(PTR)EPIRKGQ
Target chain:    B
Binder length:   100
Working dir:     /net/scratch/magnusb/43_workshop/ptm_foundry/examples/workshop_outputs/mult
RFD3 sweep:      diffusion_batch_size=3, n_batches=1
LigandMPNN:      4 sequences per backbone
RF3 sweep:       diffusion_batch_size=1


## 2. Spoof the PTM CIF on the Fly

Since we need an initial bondgraph we regenerate the phosphopeptide target directly from the sequence each time so the PTM chemistry is explicit and reproducible.

To keep the flow close to the original notebook, we first build the cifutils-style component dictionary from the sequence and then pass that dictionary into `spoof_cif_from_dictionary(...)`.

The peptide is fixed in sequence because chain `B` comes from this input CIF and later `LigandMPNN` only designs chain `A`. The peptide can still change in structure because `select_fixed_atoms` stays `false`, so RFD3 is allowed to move the coordinates even while the residue identities and PTM chemistry stay fixed.


In [6]:
spoof_input = { 'name': 'ptr_workshop',
                'components': [{'seq': 'PVPNPD(PTR)EPIRKGQ', 'chain_id': 'B'}]}

In [7]:
atom_array, component_list = components_to_atom_array(
    spoof_input["components"],
    return_components=True,
    bonds=spoof_input.get("bonds"),
)

cif_path = WORK_DIR / f"{spoof_input['name']}.cif"
RFD3_JSON_PATH = WORK_DIR / f"{spoof_input['name']}.json"

save_path = Path(
    to_cif_file(
        atom_array,
        cif_path,
        file_type="cif",
    )
)

print(f"Spoofed CIF:   {cif_path}")

Spoofed CIF:   /net/scratch/magnusb/43_workshop/ptm_foundry/examples/workshop_outputs/mult/ptr_workshop.cif


In [8]:
atom_array

AtomArray([
	Atom(np.array([nan, nan, nan], dtype=float32), chain_id="B", res_id=1, ins_code="", res_name="PRO", hetero=False, atom_name="N", element="N", is_backbone_atom=True, charge=0, stereo="N", alt_atom_id="N", atomic_number=7, chain_type=6, transformation_id="1", b_factor=nan, occupancy=1.0, is_aromatic=False, atom_id=1, is_polymer=True, chain_iid="B_1", pn_unit_id="B", molecule_id=0, chain_entity=0, pn_unit_entity=0, molecule_entity=0, pn_unit_iid="B_1", molecule_iid=0),
	Atom(np.array([nan, nan, nan], dtype=float32), chain_id="B", res_id=1, ins_code="", res_name="PRO", hetero=False, atom_name="CA", element="C", is_backbone_atom=True, charge=0, stereo="S", alt_atom_id="CA", atomic_number=6, chain_type=6, transformation_id="1", b_factor=nan, occupancy=1.0, is_aromatic=False, atom_id=2, is_polymer=True, chain_iid="B_1", pn_unit_id="B", molecule_id=0, chain_entity=0, pn_unit_entity=0, molecule_entity=0, pn_unit_iid="B_1", molecule_iid=0),
	Atom(np.array([nan, nan, nan], dtype=floa

## 3. Inspect the Bond Graph

Before running design, it helps to look at the chemistry from a few angles. The next cells show:

- a 2D atom graph with the bond types written on each edge


In [9]:
bond_array = atom_array.bonds.as_array()

In [10]:
pairs = bond_array[:, :2]                               # atom index pairs
bond_types = bond_array[:, 2]                           # bond order/type

bond_mat = atom_array.bonds.bond_type_matrix()          # bond types, -1 if no bond

In [11]:
display_mat = ~bond_mat

atom_labels = np.array([
    f"{i}: {atom_array.chain_id[i]}{int(atom_array.res_id[i])} "
    f"{atom_array.res_name[i]}:{atom_array.atom_name[i]}"
    for i in range(atom_array.array_length())
], dtype=object)

bond_type_mat = atom_array.bonds.bond_type_matrix()
bond_type_map = {
    -1: "no bond",
    0: "any",
    1: "single",
    2: "double",
    3: "triple",
    4: "quadruple",
    5: "aromatic single",
    6: "aromatic double",
    7: "aromatic triple",
    8: "coordination",
    9: "aromatic",
}
bond_type_labels = np.vectorize(lambda x: bond_type_map.get(int(x), f"unknown ({x})"))(bond_type_mat)

fig = px.imshow(
    display_mat,
    x=atom_labels,
    y=atom_labels,
    origin="lower",
)

fig.update_traces(
    text=bond_type_labels,
    hovertemplate=(
        "atom i: %{y}<br>"
        "atom j: %{x}<br>"
        "bond type: %{text}<br>"
        "shown value: %{z}<extra></extra>"
    )
)

fig.update_layout(
    width=800,
    height=800,
)

fig.update_xaxes(showticklabels=False)

fig.show()

## 4. Exercise: Write the RFD3 JSON

Edit the `json_data` cell above to finish the selector fields for a `100` residue binder against the phosphorylated peptide.

Use the bond tables and graphs above to decide what goes into:
- `select_hotspots`: atoms that should anchor interface orientation around the PTM.
- `select_buried`: atoms you want packed against the binder.
- `select_hbond_acceptor`: atoms that should accept H-bonds from the binder.

Keep the rest of the JSON scaffold unchanged.

About `OH`: in `PTR`, `OH` is the tyrosine side-chain oxygen that links the aromatic ring to the phosphate. It is part of the phosphotyrosine residue, not a separate free hydroxyl group floating off the PTM.


In [12]:
json_data = {
    EXAMPLE_NAME: {
        "input": str(cif_path),
        "contig": f"100-100,/0,B1-14",
        "dialect": 2,
        "infer_ori_strategy": "hotspots",
        "redesign_motif_sidechains": False,
        "select_fixed_atoms": False,
        "select_hotspots": {
                  "B7": "P,O1P,O2P,O3P,OH"# Fill this in.
        },
        "select_buried": {
            # Fill this in.
        },
        "select_hbond_acceptor": {
            # Fill this in.
        },
    }
}


In [13]:
print(json.dumps(json_data, indent=2))

with open(RFD3_JSON_PATH, "w") as f:
    json.dump(json_data, f, indent=2)


{
  "ptr_workshop_mult": {
    "input": "/net/scratch/magnusb/43_workshop/ptm_foundry/examples/workshop_outputs/mult/ptr_workshop.cif",
    "contig": "100-100,/0,B1-14",
    "dialect": 2,
    "infer_ori_strategy": "hotspots",
    "redesign_motif_sidechains": false,
    "select_fixed_atoms": false,
    "select_hotspots": {
      "B7": "P,O1P,O2P,O3P,OH"
    },
    "select_buried": {},
    "select_hbond_acceptor": {}
  }
}


## 5. Generate Multiple Binder Backbones with RFD3

Instead of keeping only one binder backbone, this notebook samples several in one RFD3 pass. Every backbone is carried forward through LigandMPNN and RF3 so the final score table covers the full sweep.


In [14]:
import torch

cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")
print(f"CUDA device count: {torch.cuda.device_count()}")
if cuda_available:
    print(f"Using device: {torch.cuda.get_device_name(0)}")
else:
    print("No CUDA device detected. The RFD3 / LigandMPNN / RF3 cells are workshop GPU steps and can be extremely slow on CPU.")


CUDA available: True
CUDA device count: 1
Using device: NVIDIA GeForce RTX 4080 SUPER


In [15]:
from rfd3.engine import RFD3InferenceConfig, RFD3InferenceEngine

seed_everything(7)

rfd3_config = RFD3InferenceConfig(
    ckpt_path=str(RFD3_CKPT),
    diffusion_batch_size=RFD3_DIFFUSION_BATCH_SIZE,
)
rfd3_engine = RFD3InferenceEngine(**rfd3_config)
rfd3_outputs = rfd3_engine.run(
    inputs=str(RFD3_JSON_PATH),
    out_dir=None,
    n_batches=RFD3_N_BATCHES,
)

rfd3_designs = []
for example_id, per_example_outputs in rfd3_outputs.items():
    for model_index, output in enumerate(per_example_outputs):
        design_name = f"{example_id}_rfd3_m{model_index}"
        binder_token_count = len(
            np.unique(output.atom_array[output.atom_array.chain_id == "A"].res_id)
        )
        rfd3_designs.append(
            {
                "design_name": design_name,
                "rfd3_example_id": example_id,
                "rfd3_model_index": model_index,
                "binder_token_count": binder_token_count,
                "atom_array": output.atom_array,
                "rfd3_output": output,
            }
        )

rfd3_summary = pd.DataFrame(
    [
        {
            "design_name": record["design_name"],
            "rfd3_example_id": record["rfd3_example_id"],
            "rfd3_model_index": record["rfd3_model_index"],
            "binder_token_count": record["binder_token_count"],
        }
        for record in rfd3_designs
    ]
)
display(rfd3_summary)
print(f"Generated {len(rfd3_designs)} RFD3 backbone complexes.")


DEBUG='release' is not a valid boolean; using default False
00:01:43 DEBUG transforms: Debug mode is on
Seed set to 7
00:01:44 INFO rfd3.engine: [rank: 0] Prevalidating design specification for example: ptr_workshop_ptr_workshop_mult
00:01:44 WARNING atomworks.io: We can't fix formal charges without building from templates, as we need to know the true number of hydrogens bonded to a given atom, not the inferred number. This may lead to occasional inaccuracies after adding inter-residue bonds. To avoid this and fix formal charges, set `add_missing_atoms = True`.
00:01:44 WARNING atomworks.io: Field is_motif_atom_unindexed not found in file, ignoring.
00:01:44 WARNING atomworks.io: Field is_motif_atom_with_fixed_coord not found in file, ignoring.
00:01:44 WARNING atomworks.io: Field is_motif_atom_with_fixed_seq not found in file, ignoring.
00:01:44 WARNING atomworks.io: Field is_motif_atom_unindexed_motif_breakpoint not found in file, ignoring.
00:01:48 WARNING atomworks.io: We can't fix

,design_name,rfd3_example_id,rfd3_model_index,binder_token_count
0,ptr_workshop_ptr_workshop_mult_0_rfd3_m0,ptr_workshop_ptr_workshop_mult_0,0,100
1,ptr_workshop_ptr_workshop_mult_0_rfd3_m1,ptr_workshop_ptr_workshop_mult_0,1,100
2,ptr_workshop_ptr_workshop_mult_0_rfd3_m2,ptr_workshop_ptr_workshop_mult_0,2,100


Generated 3 RFD3 backbone complexes.


### Studio-style RFD3 Browser

This browser keeps the structure and scatter plot side by side. Use the metric dropdowns to switch the RFD3 step scatter axes while paging through the generated backbones on the left.


In [16]:
RFD3_BROWSER_METRIC_LABELS = {
    "rfd3_model_index": "RFD3 model index",
    "binder_token_count": "Binder residues",
    "atom_count": "Total atoms",
    "binder_atom_count": "Binder atoms",
    "target_atom_count": "Target atoms",
    "binder_backbone_rmsd_to_first": "Binder-backbone RMSD to first (A)",
    "binder_min_ptr_distance": "Minimum binder-PTR distance (A)",
    "binder_backbone_centroid_to_ptr": "Binder-backbone centroid to PTR (A)",
}

rfd3_browser_df = build_rfd3_metrics_df(
    rfd3_designs,
    rfd3_summary,
    binder_chain_id="A",
    target_chain_id=TARGET_CHAIN_ID,
    ptr_residue_id=7,
)

display(
    make_studio_metric_browser(
        records=rfd3_designs,
        metrics_df=rfd3_browser_df,
        structure_factory=lambda record: make_browser_structure_spec(
            record["atom_array"],
            ptr_chain=TARGET_CHAIN_ID,
            ptr_resi=7,
            width=520,
            height=430,
        ),
        label_column="design_name",
        title="RFD3 candidate metric explorer",
        metric_columns=[
            "rfd3_model_index",
            "binder_token_count",
            "atom_count",
            "binder_atom_count",
            "target_atom_count",
            "binder_backbone_rmsd_to_first",
            "binder_min_ptr_distance",
            "binder_backbone_centroid_to_ptr",
        ],
        default_x="binder_backbone_rmsd_to_first",
        default_y="binder_min_ptr_distance",
        default_z="binder_token_count",
        metric_labels=RFD3_BROWSER_METRIC_LABELS,
        note_html="Browse RFD3 backbones in Mol* with the scatter on the right.",
    )
)


## 6. Design Binder Sequences for Every Backbone

LigandMPNN now runs on every RFD3 backbone instead of just the first one. Each backbone gets its own small sequence sweep so the final comparison keeps both backbone diversity and sequence diversity.


In [17]:
from mpnn.inference_engines.mpnn import MPNNInferenceEngine

mpnn_engine = MPNNInferenceEngine(
    model_type="ligand_mpnn",
    checkpoint_path=str(LIGANDMPNN_CKPT),
    is_legacy_weights=True,
    out_directory=None,
    write_structures=False,
    write_fasta=False,
)

mpnn_input_dicts = [
    {
        "name": record["design_name"],
        "batch_size": MPNN_SEQUENCES_PER_BACKBONE,
        "remove_waters": True,
        "designed_chains": ["A"],
    }
    for record in rfd3_designs
]

mpnn_outputs = mpnn_engine.run(
    input_dicts=mpnn_input_dicts,
    atom_arrays=[record["atom_array"] for record in rfd3_designs],
)

rfd3_by_name = {record["design_name"]: record for record in rfd3_designs}
mpnn_designs = []
for output in mpnn_outputs:
    parent_name = output.input_dict["name"]
    design_idx = int(output.output_dict["design_idx"])
    parent_record = rfd3_by_name[parent_name]
    mpnn_name = f"{parent_name}_mpnn_d{design_idx}"
    interface_recovery = output.output_dict["ligand_interface_sequence_recovery"]
    mpnn_designs.append(
        {
            "design_name": mpnn_name,
            "parent_name": parent_name,
            "rfd3_example_id": parent_record["rfd3_example_id"],
            "rfd3_model_index": parent_record["rfd3_model_index"],
            "rfd3_complex": parent_record["atom_array"],
            "mpnn_design_idx": design_idx,
            "mpnn_output": output,
            "reference_complex": output.atom_array,
            "binder_sequence": extract_chain_sequence(output.atom_array, "A"),
            "target_sequence": extract_chain_sequence(output.atom_array, TARGET_CHAIN_ID),
            "sequence_recovery": float(output.output_dict["sequence_recovery"]),
            "ligand_interface_sequence_recovery": (
                float(interface_recovery) if interface_recovery is not None else np.nan
            ),
        }
    )

mpnn_summary = pd.DataFrame(
    [
        {
            "design_name": record["design_name"],
            "parent_name": record["parent_name"],
            "rfd3_model_index": record["rfd3_model_index"],
            "mpnn_design_idx": record["mpnn_design_idx"],
            "binder_sequence": record["binder_sequence"],
            "sequence_recovery": record["sequence_recovery"],
            "ligand_interface_sequence_recovery": record["ligand_interface_sequence_recovery"],
        }
        for record in mpnn_designs
    ]
)
display(mpnn_summary)
print(f"Generated {len(mpnn_designs)} LigandMPNN sequence designs across {len(rfd3_designs)} backbones.")


,design_name,parent_name,rfd3_model_index,mpnn_design_idx,binder_sequence,sequence_recovery,ligand_interface_sequence_recovery
0,ptr_workshop_ptr_workshop_mult_0_rfd3_m0_mpnn_d0,ptr_workshop_ptr_workshop_mult_0_rfd3_m0,0,0,TWREVDWPADLRTALDGAAAHLGLPPVPEKALSTEDGSLYVPAGSR...,0.46,0.666667
1,ptr_workshop_ptr_workshop_mult_0_rfd3_m0_mpnn_d1,ptr_workshop_ptr_workshop_mult_0_rfd3_m0,0,1,TWTEVSWPPDLRRALDGAAAHLGLPPVPERALATADGSLYRPAGSE...,0.45,0.333333
2,ptr_workshop_ptr_workshop_mult_0_rfd3_m0_mpnn_d2,ptr_workshop_ptr_workshop_mult_0_rfd3_m0,0,2,TWREAEWPPDARTALDGAMAYLGLPPVPARALTNEDGSLWVPAGSV...,0.43,0.333333
3,ptr_workshop_ptr_workshop_mult_0_rfd3_m0_mpnn_d3,ptr_workshop_ptr_workshop_mult_0_rfd3_m0,0,3,TWREAEWPPAARTALDGAAAHLGLPPVPARALTTEDGSLWVPAGAE...,0.47,0.666667
4,ptr_workshop_ptr_workshop_mult_0_rfd3_m1_mpnn_d0,ptr_workshop_ptr_workshop_mult_0_rfd3_m1,1,0,VTTPAEFCDALAAGTLRTSIPSSQLRLRTDPETGRGEILARGASTV...,0.51,0.444444
5,ptr_workshop_ptr_workshop_mult_0_rfd3_m1_mpnn_d1,ptr_workshop_ptr_workshop_mult_0_rfd3_m1,1,1,VTTLAEFVEALDAGTLQLSIPSSQLSLRTDPETGKGMLRASGASTL...,0.48,0.444444
6,ptr_workshop_ptr_workshop_mult_0_rfd3_m1_mpnn_d2,ptr_workshop_ptr_workshop_mult_0_rfd3_m1,1,2,VRTLAEFCEALAAGTLRLDTPESQLRLRTDPETGRGMLLAEGASAR...,0.58,0.444444
7,ptr_workshop_ptr_workshop_mult_0_rfd3_m1_mpnn_d3,ptr_workshop_ptr_workshop_mult_0_rfd3_m1,1,3,VKTLAEACEALDAGTLKTSIPSSQLRLRTDPATGRGAILAEGASAT...,0.59,0.444444
8,ptr_workshop_ptr_workshop_mult_0_rfd3_m2_mpnn_d0,ptr_workshop_ptr_workshop_mult_0_rfd3_m2,2,0,APTARERELAALAARAASEALGLPVRAFVNEEGIAAAEASDRTLIV...,0.51,0.500000
9,ptr_workshop_ptr_workshop_mult_0_rfd3_m2_mpnn_d1,ptr_workshop_ptr_workshop_mult_0_rfd3_m2,2,1,APTAEELALAAAAAAAASAALGLPVSAFVGEEGIAAARASTRTLIV...,0.56,1.000000


Generated 12 LigandMPNN sequence designs across 3 backbones.


## 7. Refold Every Designed Complex with RF3

RF3 now refolds the whole LigandMPNN candidate set so we can score every binder-target complex instead of manually picking one design up front.


In [18]:
from rf3.inference_engines.rf3 import RF3InferenceEngine
from rf3.utils.inference import InferenceInput

rf3_engine = RF3InferenceEngine(
    ckpt_path=str(RF3_CKPT),
    diffusion_batch_size=RF3_DIFFUSION_BATCH_SIZE,
    verbose=False,
)

rf3_inputs = [
    InferenceInput.from_atom_array(
        record["reference_complex"],
        example_id=record["design_name"],
    )
    for record in mpnn_designs
]

rf3_outputs = rf3_engine.run(
    inputs=rf3_inputs,
    annotate_b_factor_with_plddt=True,
)

rf3_designs = []
for record in mpnn_designs:
    example_id = record["design_name"]
    per_example_outputs = rf3_outputs[example_id]
    for sample_idx, rf3_output in enumerate(per_example_outputs):
        rf3_name = example_id if len(per_example_outputs) == 1 else f"{example_id}_rf3_s{sample_idx}"
        rf3_designs.append(
            {
                **record,
                "rf3_name": rf3_name,
                "rf3_sample_idx": sample_idx,
                "rf3_output": rf3_output,
                "summary_confidences": rf3_output.summary_confidences,
            }
        )

rf3_summary = pd.DataFrame(
    [
        {
            "rf3_name": record["rf3_name"],
            "design_name": record["design_name"],
            "overall_plddt": float(record["summary_confidences"].get("overall_plddt", np.nan)),
            "overall_pae": float(record["summary_confidences"].get("overall_pae", np.nan)),
            "ptm": (
                float(record["summary_confidences"].get("ptm"))
                if record["summary_confidences"].get("ptm") is not None
                else np.nan
            ),
            "iptm": (
                float(record["summary_confidences"].get("iptm"))
                if record["summary_confidences"].get("iptm") is not None
                else np.nan
            ),
            "ranking_score": (
                float(record["summary_confidences"].get("ranking_score"))
                if record["summary_confidences"].get("ranking_score") is not None
                else np.nan
            ),
        }
        for record in rf3_designs
    ]
)
display(rf3_summary)
print(f"Ran RF3 on {len(rf3_designs)} designed complexes.")


00:02:08 WARNING atomworks.io: The `extra_fields` argument will be ignored if there is no CIF file input.
00:02:08 WARNING atomworks.io: Adding missing atoms will erase extra fields. If you just want to load a structure with the given extra fields, you should probably use the much faster 'load_any' function from atomworks.io.utils.io_utils instead of 'parse'. Parse is meant for cleaning up structures from the RCSB PDB.
00:02:08 WARNING atomworks.io: The `extra_fields` argument will be ignored if there is no CIF file input.
00:02:08 WARNING atomworks.io: Adding missing atoms will erase extra fields. If you just want to load a structure with the given extra fields, you should probably use the much faster 'load_any' function from atomworks.io.utils.io_utils instead of 'parse'. Parse is meant for cleaning up structures from the RCSB PDB.
00:02:08 WARNING atomworks.io: The `extra_fields` argument will be ignored if there is no CIF file input.
00:02:08 WARNING atomworks.io: Adding missing at

00:02:08 WARNING atomworks.io: The `extra_fields` argument will be ignored if there is no CIF file input.
00:02:08 WARNING atomworks.io: Adding missing atoms will erase extra fields. If you just want to load a structure with the given extra fields, you should probably use the much faster 'load_any' function from atomworks.io.utils.io_utils instead of 'parse'. Parse is meant for cleaning up structures from the RCSB PDB.
00:02:08 WARNING atomworks.io: The `extra_fields` argument will be ignored if there is no CIF file input.
00:02:08 WARNING atomworks.io: Adding missing atoms will erase extra fields. If you just want to load a structure with the given extra fields, you should probably use the much faster 'load_any' function from atomworks.io.utils.io_utils instead of 'parse'. Parse is meant for cleaning up structures from the RCSB PDB.
00:02:08 WARNING atomworks.io: The `extra_fields` argument will be ignored if there is no CIF file input.
00:02:08 WARNING atomworks.io: Adding missing at

,rf3_name,design_name,overall_plddt,overall_pae,ptm,iptm,ranking_score
0,ptr_workshop_ptr_workshop_mult_0_rfd3_m0_mpnn_d0,ptr_workshop_ptr_workshop_mult_0_rfd3_m0_mpnn_d0,0.7620,7.3307,0.749478,0.497196,0.5477
1,ptr_workshop_ptr_workshop_mult_0_rfd3_m0_mpnn_d1,ptr_workshop_ptr_workshop_mult_0_rfd3_m0_mpnn_d1,0.8219,5.2937,0.848852,0.623987,0.6690
2,ptr_workshop_ptr_workshop_mult_0_rfd3_m0_mpnn_d2,ptr_workshop_ptr_workshop_mult_0_rfd3_m0_mpnn_d2,0.7896,5.6686,0.825236,0.798566,0.8039
3,ptr_workshop_ptr_workshop_mult_0_rfd3_m0_mpnn_d3,ptr_workshop_ptr_workshop_mult_0_rfd3_m0_mpnn_d3,0.8197,4.2436,0.885809,0.761220,0.7861
4,ptr_workshop_ptr_workshop_mult_0_rfd3_m1_mpnn_d0,ptr_workshop_ptr_workshop_mult_0_rfd3_m1_mpnn_d0,0.7571,7.2045,0.723724,0.673035,0.6832
5,ptr_workshop_ptr_workshop_mult_0_rfd3_m1_mpnn_d1,ptr_workshop_ptr_workshop_mult_0_rfd3_m1_mpnn_d1,0.8185,4.8365,0.870381,0.724783,0.7539
6,ptr_workshop_ptr_workshop_mult_0_rfd3_m1_mpnn_d2,ptr_workshop_ptr_workshop_mult_0_rfd3_m1_mpnn_d2,0.7910,6.6565,0.805625,0.514448,0.5727
7,ptr_workshop_ptr_workshop_mult_0_rfd3_m1_mpnn_d3,ptr_workshop_ptr_workshop_mult_0_rfd3_m1_mpnn_d3,0.7761,6.0432,0.762081,0.657149,0.6781
8,ptr_workshop_ptr_workshop_mult_0_rfd3_m2_mpnn_d0,ptr_workshop_ptr_workshop_mult_0_rfd3_m2_mpnn_d0,0.8337,4.9273,0.871655,0.671997,0.7119
9,ptr_workshop_ptr_workshop_mult_0_rfd3_m2_mpnn_d1,ptr_workshop_ptr_workshop_mult_0_rfd3_m2_mpnn_d1,0.8449,4.6512,0.880277,0.719913,0.7520


Ran RF3 on 12 designed complexes.


### Studio-style RF3 Browser

This RF3 explorer mirrors the Studio flow more closely: the current complex is shown on the left, and the right-hand scatter can be re-plotted with any of the RF3 confidence metrics from this step.


In [19]:
RF3_BROWSER_METRIC_LABELS = {
    "overall_plddt": "Overall pLDDT",
    "overall_plddt_pct": "Overall pLDDT (%)",
    "overall_pae": "Overall PAE (A)",
    "min_pae": "Minimum interface PAE (A)",
    "ptm": "pTM",
    "iptm": "ipTM",
    "ranking_score": "Ranking score",
    "rf3_sample_idx": "RF3 sample index",
}

rf3_browser_df = build_rf3_metrics_df(rf3_designs, rf3_summary)


def _make_rf3_aligned_spec(record):
    reference = record["reference_complex"]
    mobile = record["rf3_output"].atom_array
    binder_mask_ref = (
        (reference.chain_id == "A")
        & np.isin(reference.atom_name, PROTEIN_BACKBONE_ATOM_NAMES)
    )
    binder_mask_mobile = (
        (mobile.chain_id == "A")
        & np.isin(mobile.atom_name, PROTEIN_BACKBONE_ATOM_NAMES)
    )
    aligned_mobile, _, _ = align_mobile_on_binder_backbone(
        reference, mobile, binder_mask_ref, binder_mask_mobile
    )
    return make_browser_structure_spec(
        reference,
        aligned_mobile,
        ptr_chain=TARGET_CHAIN_ID,
        ptr_resi=7,
        width=520,
        height=430,
    )


display(
    make_studio_metric_browser(
        records=rf3_designs,
        metrics_df=rf3_browser_df,
        structure_factory=_make_rf3_aligned_spec,
        label_column="rf3_name",
        title="RF3 confidence explorer",
        metric_columns=[
            "rf3_sample_idx",
            "overall_plddt",
            "overall_plddt_pct",
            "overall_pae",
            "min_pae",
            "ptm",
            "iptm",
            "ranking_score",
        ],
        default_x="min_pae",
        default_y="overall_plddt_pct",
        default_z="ranking_score",
        metric_labels=RF3_BROWSER_METRIC_LABELS,
        note_html="Gray = LigandMPNN design, red = RF3 refold (binder-backbone aligned). Click a scatter point to jump to that structure.",
    )
)


## 8. Score Every Candidate

The helper functions below are the same notebook checks as the single-design workshop, but now they run across the full candidate set so the last table has one row per refolded complex.


In [20]:
FINAL_FILTER_MAX_PEPTIDE_CA_RMSD = 1.5
FINAL_FILTER_MIN_PO4_BURIAL = 0.35
FINAL_FILTER_MIN_HBONDS = 2

ptm_residue_id = 7


In [21]:
def compute_candidate_metrics(record):
    reference_complex = record["reference_complex"]
    mobile_complex = record["rf3_output"].atom_array

    binder_backbone_mask_ref = (
        (reference_complex.chain_id == "A")
        & np.isin(reference_complex.atom_name, PROTEIN_BACKBONE_ATOM_NAMES)
    )
    binder_backbone_mask_mobile = (
        (mobile_complex.chain_id == "A")
        & np.isin(mobile_complex.atom_name, PROTEIN_BACKBONE_ATOM_NAMES)
    )

    peptide_mask_ref = reference_complex.chain_id == TARGET_CHAIN_ID
    peptide_mask_mobile = mobile_complex.chain_id == TARGET_CHAIN_ID

    peptide_ca_mask_ref = peptide_mask_ref & (reference_complex.atom_name == "CA")
    peptide_ca_mask_mobile = peptide_mask_mobile & (mobile_complex.atom_name == "CA")

    ptr_mask_ref = (
        peptide_mask_ref
        & (reference_complex.res_id == ptm_residue_id)
        & (reference_complex.res_name == "PTR")
    )
    ptr_mask_mobile = (
        peptide_mask_mobile
        & (mobile_complex.res_id == ptm_residue_id)
        & (mobile_complex.res_name == "PTR")
    )

    po4_mask_ref = ptr_mask_ref & np.isin(reference_complex.atom_name, PHOSPHATE_ATOMS)
    po4_mask_mobile = ptr_mask_mobile & np.isin(mobile_complex.atom_name, PHOSPHATE_ATOMS)

    aligned_rf3_complex, binder_transform, binder_alignment_rmsd = align_mobile_on_binder_backbone(
        reference_complex,
        mobile_complex,
        binder_backbone_mask_ref,
        binder_backbone_mask_mobile,
    )

    binder_ref_idx, binder_mobile_idx = paired_common_indices(
        reference_complex,
        mobile_complex,
        binder_backbone_mask_ref,
        binder_backbone_mask_mobile,
    )

    rmsd_rows = [
        {
            "metric": "binder_backbone_alignment_rmsd",
            "rmsd_angstrom": binder_alignment_rmsd,
            "paired_atoms": len(binder_ref_idx),
        }
    ]

    for metric_name, ref_mask, mobile_mask in [
        ("whole_peptide_ca_rmsd", peptide_ca_mask_ref, peptide_ca_mask_mobile),
        ("whole_peptide_all_atom_rmsd", peptide_mask_ref, peptide_mask_mobile),
        ("ptr_all_atom_rmsd", ptr_mask_ref, ptr_mask_mobile),
        ("po4_only_rmsd", po4_mask_ref, po4_mask_mobile),
    ]:
        metric_rmsd, paired_atoms = rmsd_for_masks(
            reference_complex,
            aligned_rf3_complex,
            ref_mask,
            mobile_mask,
            allow_mismatch=True,
        )
        rmsd_rows.append(
            {
                "metric": metric_name,
                "rmsd_angstrom": metric_rmsd,
                "paired_atoms": paired_atoms,
            }
        )

    rmsd_metrics = pd.DataFrame(rmsd_rows)
    rmsd_lookup = dict(zip(rmsd_metrics["metric"], rmsd_metrics["rmsd_angstrom"]))

    hbond_metrics = compute_phosphosite_hbond_metrics(
        aligned_rf3_complex,
        chain_id=TARGET_CHAIN_ID,
        residue_id=ptm_residue_id,
        res_name="PTR",
    )
    po4_sasa_metrics = compute_selection_sasa_metrics(aligned_rf3_complex, po4_mask_mobile)
    ptr_sasa_metrics = compute_selection_sasa_metrics(aligned_rf3_complex, ptr_mask_mobile)

    summary = record["summary_confidences"]
    overall_plddt = float(summary.get("overall_plddt", np.nan))
    po4_fraction_buried = po4_sasa_metrics["fraction_buried"]
    phosphosite_hbonds = hbond_metrics["phosphosite_hbonds"]

    metric_row = {
        "rf3_name": record["rf3_name"],
        "design_name": record["design_name"],
        "parent_name": record["parent_name"],
        "rfd3_example_id": record["rfd3_example_id"],
        "rfd3_model_index": record["rfd3_model_index"],
        "mpnn_design_idx": record["mpnn_design_idx"],
        "binder_sequence": record["binder_sequence"],
        "sequence_recovery": record["sequence_recovery"],
        "ligand_interface_sequence_recovery": record["ligand_interface_sequence_recovery"],
        "overall_plddt": overall_plddt,
        "overall_plddt_pct": overall_plddt * 100 if np.isfinite(overall_plddt) else np.nan,
        "overall_pae": float(summary.get("overall_pae", np.nan)),
        "min_pae": extract_min_interface_pae(summary),
        "ptm": float(summary.get("ptm", np.nan)) if summary.get("ptm") is not None else np.nan,
        "iptm": float(summary.get("iptm", np.nan)) if summary.get("iptm") is not None else np.nan,
        "ranking_score": float(summary.get("ranking_score", np.nan)) if summary.get("ranking_score") is not None else np.nan,
        "binder_backbone_alignment_rmsd": rmsd_lookup["binder_backbone_alignment_rmsd"],
        "peptide_ca_rmsd": rmsd_lookup["whole_peptide_ca_rmsd"],
        "peptide_all_atom_rmsd": rmsd_lookup["whole_peptide_all_atom_rmsd"],
        "ptr_all_atom_rmsd": rmsd_lookup["ptr_all_atom_rmsd"],
        "po4_only_rmsd": rmsd_lookup["po4_only_rmsd"],
        "po4_fraction_buried": po4_fraction_buried,
        "ptr_fraction_buried": ptr_sasa_metrics["fraction_buried"],
        "phosphosite_hbonds": phosphosite_hbonds,
        "peptide_ca_pass": rmsd_lookup["whole_peptide_ca_rmsd"] < FINAL_FILTER_MAX_PEPTIDE_CA_RMSD,
        "po4_burial_pass": po4_fraction_buried > FINAL_FILTER_MIN_PO4_BURIAL,
        "phosphosite_hbonds_pass": phosphosite_hbonds >= FINAL_FILTER_MIN_HBONDS,
        "overall_tutorial_pass": (
            rmsd_lookup["whole_peptide_ca_rmsd"] < FINAL_FILTER_MAX_PEPTIDE_CA_RMSD
            and po4_fraction_buried > FINAL_FILTER_MIN_PO4_BURIAL
            and phosphosite_hbonds >= FINAL_FILTER_MIN_HBONDS
        ),
    }

    detail = {
        **record,
        "aligned_rf3_complex": aligned_rf3_complex,
        "hbond_metrics": hbond_metrics,
        "po4_sasa_metrics": po4_sasa_metrics,
        "ptr_sasa_metrics": ptr_sasa_metrics,
        "metric_row": metric_row,
    }
    return metric_row, detail

scored_rows = []
scored_details = []
for record in rf3_designs:
    metric_row, detail = compute_candidate_metrics(record)
    scored_rows.append(metric_row)
    scored_details.append(detail)

all_scores_df = pd.DataFrame(scored_rows).sort_values(
    by=["overall_tutorial_pass", "overall_plddt", "min_pae"],
    ascending=[False, False, True],
).reset_index(drop=True)

detail_by_name = {detail["rf3_name"]: detail for detail in scored_details}
scored_details = [detail_by_name[name] for name in all_scores_df["rf3_name"]]


In [22]:
display(all_scores_df.round(3))
print(f"Scored {len(all_scores_df)} RF3-refolded designs.")


,rf3_name,design_name,parent_name,rfd3_example_id,rfd3_model_index,mpnn_design_idx,binder_sequence,sequence_recovery,ligand_interface_sequence_recovery,overall_plddt,...,peptide_all_atom_rmsd,ptr_all_atom_rmsd,po4_only_rmsd,po4_fraction_buried,ptr_fraction_buried,phosphosite_hbonds,peptide_ca_pass,po4_burial_pass,phosphosite_hbonds_pass,overall_tutorial_pass
0,ptr_workshop_ptr_workshop_mult_0_rfd3_m2_mpnn_d3,ptr_workshop_ptr_workshop_mult_0_rfd3_m2_mpnn_d3,ptr_workshop_ptr_workshop_mult_0_rfd3_m2,ptr_workshop_ptr_workshop_mult_0,2,3,APTAEERALAALAAAAASAALGLPVRAFVGEEGIAAAEASTRTLIV...,0.52,0.000,0.850,...,20.936,15.122,18.648,0.909,0.891,3.0,False,True,True,False
1,ptr_workshop_ptr_workshop_mult_0_rfd3_m2_mpnn_d1,ptr_workshop_ptr_workshop_mult_0_rfd3_m2_mpnn_d1,ptr_workshop_ptr_workshop_mult_0_rfd3_m2,ptr_workshop_ptr_workshop_mult_0,2,1,APTAEELALAAAAAAAASAALGLPVSAFVGEEGIAAARASTRTLIV...,0.56,1.000,0.845,...,18.111,10.025,12.791,0.449,0.658,2.0,False,True,True,False
2,ptr_workshop_ptr_workshop_mult_0_rfd3_m2_mpnn_d2,ptr_workshop_ptr_workshop_mult_0_rfd3_m2_mpnn_d2,ptr_workshop_ptr_workshop_mult_0_rfd3_m2,ptr_workshop_ptr_workshop_mult_0,2,2,APTAEQEALAAAAAAAASAALGLPVSAFVGEAGIAAARASTRTLIV...,0.54,0.500,0.840,...,18.097,13.147,17.011,0.907,0.868,3.0,False,True,True,False
3,ptr_workshop_ptr_workshop_mult_0_rfd3_m2_mpnn_d0,ptr_workshop_ptr_workshop_mult_0_rfd3_m2_mpnn_d0,ptr_workshop_ptr_workshop_mult_0_rfd3_m2,ptr_workshop_ptr_workshop_mult_0,2,0,APTARERELAALAARAASEALGLPVRAFVNEEGIAAAEASDRTLIV...,0.51,0.500,0.834,...,16.868,8.335,11.378,0.468,0.730,2.0,False,True,True,False
4,ptr_workshop_ptr_workshop_mult_0_rfd3_m0_mpnn_d1,ptr_workshop_ptr_workshop_mult_0_rfd3_m0_mpnn_d1,ptr_workshop_ptr_workshop_mult_0_rfd3_m0,ptr_workshop_ptr_workshop_mult_0,0,1,TWTEVSWPPDLRRALDGAAAHLGLPPVPERALATADGSLYRPAGSE...,0.45,0.333,0.822,...,15.720,5.435,4.136,0.932,0.961,3.0,False,True,True,False
5,ptr_workshop_ptr_workshop_mult_0_rfd3_m0_mpnn_d3,ptr_workshop_ptr_workshop_mult_0_rfd3_m0_mpnn_d3,ptr_workshop_ptr_workshop_mult_0_rfd3_m0,ptr_workshop_ptr_workshop_mult_0,0,3,TWREAEWPPAARTALDGAAAHLGLPPVPARALTTEDGSLWVPAGAE...,0.47,0.667,0.820,...,18.188,7.100,2.974,0.826,0.839,1.0,False,True,False,False
6,ptr_workshop_ptr_workshop_mult_0_rfd3_m1_mpnn_d1,ptr_workshop_ptr_workshop_mult_0_rfd3_m1_mpnn_d1,ptr_workshop_ptr_workshop_mult_0_rfd3_m1,ptr_workshop_ptr_workshop_mult_0,1,1,VTTLAEFVEALDAGTLQLSIPSSQLSLRTDPETGKGMLRASGASTL...,0.48,0.444,0.818,...,3.776,3.417,3.670,0.948,0.874,1.0,False,True,False,False
7,ptr_workshop_ptr_workshop_mult_0_rfd3_m1_mpnn_d2,ptr_workshop_ptr_workshop_mult_0_rfd3_m1_mpnn_d2,ptr_workshop_ptr_workshop_mult_0_rfd3_m1,ptr_workshop_ptr_workshop_mult_0,1,2,VRTLAEFCEALAAGTLRLDTPESQLRLRTDPETGRGMLLAEGASAR...,0.58,0.444,0.791,...,14.126,8.321,4.438,0.950,0.917,5.0,False,True,True,False
8,ptr_workshop_ptr_workshop_mult_0_rfd3_m0_mpnn_d2,ptr_workshop_ptr_workshop_mult_0_rfd3_m0_mpnn_d2,ptr_workshop_ptr_workshop_mult_0_rfd3_m0,ptr_workshop_ptr_workshop_mult_0,0,2,TWREAEWPPDARTALDGAMAYLGLPPVPARALTNEDGSLWVPAGSV...,0.43,0.333,0.790,...,13.767,10.756,14.266,0.551,0.742,1.0,False,True,False,False
9,ptr_workshop_ptr_workshop_mult_0_rfd3_m1_mpnn_d3,ptr_workshop_ptr_workshop_mult_0_rfd3_m1_mpnn_d3,ptr_workshop_ptr_workshop_mult_0_rfd3_m1,ptr_workshop_ptr_workshop_mult_0,1,3,VKTLAEACEALDAGTLKTSIPSSQLRLRTDPATGRGAILAEGASAT...,0.59,0.444,0.776,...,22.179,13.746,6.696,0.896,0.917,5.0,False,True,True,False


Scored 12 RF3-refolded designs.


In [23]:
scatter_fig = px.scatter(
    all_scores_df,
    x="min_pae",
    y="overall_plddt_pct",
    color="overall_tutorial_pass",
    hover_name="rf3_name",
    hover_data=[
        "rfd3_model_index",
        "mpnn_design_idx",
        "overall_pae",
        "ranking_score",
        "sequence_recovery",
        "phosphosite_hbonds",
        "po4_fraction_buried",
    ],
    labels={
        "min_pae": "Minimum interface PAE (A)",
        "overall_plddt_pct": "Overall pLDDT (%)",
        "overall_tutorial_pass": "Workshop pass",
    },
    title="RF3 confidence across all generated complexes",
)
scatter_fig.update_traces(marker={"size": 11, "line": {"width": 1, "color": "white"}})
scatter_fig.update_layout(legend_title_text="Workshop pass")
scatter_fig.show()


## 9. Browse and Compare the Generated Structures


In [24]:
FINAL_BROWSER_METRIC_LABELS = {
    "overall_plddt_pct": "Overall pLDDT (%)",
    "min_pae": "Minimum interface PAE (A)",
    "overall_pae": "Overall PAE (A)",
    "ranking_score": "Ranking score",
    "sequence_recovery": "Sequence recovery",
    "ligand_interface_sequence_recovery": "Interface sequence recovery",
    "binder_backbone_alignment_rmsd": "Binder-backbone alignment RMSD (A)",
    "peptide_ca_rmsd": "Peptide CA RMSD (A)",
    "peptide_all_atom_rmsd": "Peptide all-atom RMSD (A)",
    "ptr_all_atom_rmsd": "PTR all-atom RMSD (A)",
    "po4_only_rmsd": "PO4-only RMSD (A)",
    "po4_fraction_buried": "PO4 fraction buried",
    "ptr_fraction_buried": "PTR fraction buried",
    "phosphosite_hbonds": "Phosphosite H-bonds",
}

display(
    make_studio_metric_browser(
        records=scored_details,
        metrics_df=all_scores_df,
        structure_factory=lambda detail: make_structure_overlay_spec(
            detail["reference_complex"],
            detail["aligned_rf3_complex"],
            ptr_chain=TARGET_CHAIN_ID,
            ptr_resi=ptm_residue_id,
            width=780,
            height=480,
        ),
        label_column="rf3_name",
        title="Final candidate explorer",
        metric_columns=[
            "overall_plddt_pct",
            "min_pae",
            "overall_pae",
            "ranking_score",
            "sequence_recovery",
            "ligand_interface_sequence_recovery",
            "binder_backbone_alignment_rmsd",
            "peptide_ca_rmsd",
            "peptide_all_atom_rmsd",
            "ptr_all_atom_rmsd",
            "po4_only_rmsd",
            "po4_fraction_buried",
            "ptr_fraction_buried",
            "phosphosite_hbonds",
        ],
        default_x="min_pae",
        default_y="overall_plddt_pct",
        default_z="ranking_score",
        metric_labels=FINAL_BROWSER_METRIC_LABELS,
        note_html="Gray = LigandMPNN design, red = RF3 refold after binder-backbone alignment. Use the X, Y, and Z dropdowns to compare final candidates without leaving the viewer.",
        table_columns=[
            "rf3_name",
            "overall_plddt_pct",
            "min_pae",
            "ranking_score",
            "sequence_recovery",
            "po4_fraction_buried",
            "phosphosite_hbonds",
        ],
    )
)


## 10. Export the Multi-design Results

This writes the aggregate score table plus the RFD3, LigandMPNN, and RF3 structures for the whole sweep into a dedicated output directory.


In [ ]:
EXPORT_DIR = WORK_DIR / "mult_results"
RFD3_EXPORT_DIR = EXPORT_DIR / "rfd3"
MPNN_EXPORT_DIR = EXPORT_DIR / "mpnn"
RF3_EXPORT_DIR = EXPORT_DIR / "rf3"
for path in [EXPORT_DIR, RFD3_EXPORT_DIR, MPNN_EXPORT_DIR, RF3_EXPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

scores_path = EXPORT_DIR / f"{EXAMPLE_NAME}_all_scores.csv"
all_scores_df.to_csv(scores_path, index=False)

for detail in scored_details:
    rfd3_base = RFD3_EXPORT_DIR / detail["design_name"]
    mpnn_base = MPNN_EXPORT_DIR / detail["design_name"]
    rf3_base = RF3_EXPORT_DIR / detail["rf3_name"]
    to_cif_file(
        detail["rfd3_complex"],
        rfd3_base,
        file_type="cif",
        include_entity_poly=False,
    )
    detail["mpnn_output"].write_structure(base_path=mpnn_base)
    to_cif_file(
        detail["rf3_output"].atom_array,
        rf3_base,
        file_type="cif",
        include_entity_poly=False,
    )

print("Saved multi-design results:")
print(f" - score table: {scores_path}")
print(f" - RFD3 structures: {RFD3_EXPORT_DIR}")
print(f" - LigandMPNN structures: {MPNN_EXPORT_DIR}")
print(f" - RF3 structures: {RF3_EXPORT_DIR}")


## What To Try Next

1. Increase `RFD3_N_BATCHES` or `MPNN_SEQUENCES_PER_BACKBONE` if you want a wider sweep.
2. Sort `all_scores_df` by different columns such as `ranking_score`, `min_pae`, or `po4_fraction_buried`.
3. Use the scatter plot and the previous/next browser together to compare low-PAE, high-pLDDT candidates by eye.
4. Save the strongest binders from `mult_results` for deeper follow-up or downstream pipeline runs.
